<a href="https://colab.research.google.com/github/bemakerorg/AIoT_Book_II_RF/blob/main/AIoT_RF_Book_ES_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ESERCIZIO 18**
Riduziuone del Modello AI con TF Lite e Verifica mantenimento previsioni.

In [ ]:
# Importiamo la libreria TensrFlow e verifichiamo che sia la versione 2.18.0
import tensorflow as tf
print("Current TensorFlow version:", {tf.__version__})
if tf.__version__ != "2.18.0":
    print(f"Current TensorFlow version: {tf.__version__}, switching to 2.18.0")

    # Disistalla la libreria se diversa dalla 2.18.0
    !pip uninstall -y tensorflow

    # Installa la libreria TensorFlow 2.18
    !pip install tensorflow==2.18

    # Dopo l'istallazione occorre riavviare il runtime
    print("TensorFlow 2.18 installed.")
    print("Please click on the Runtime > Restart session and run all.")
else:
    print("TensorFlow 2.18 is already installed.")

In [ ]:
# 1) Import delle librerie necessarie
import numpy as np                                       # NumPy per gli array e operazioni numeriche
from tensorflow import keras                             # Keras API per costruire modelli di reti
import pathlib                                           # pathlib per operazioni su percorsi di file
import matplotlib.pyplot as plt                          # Importiamo la Libreria per il plottaggio molto utile per la visualizzazione

In [ ]:
# 2) Definizione del modello di rete neurale multistrato e multineurone - piramide decrescente
model = keras.Sequential([
    keras.layers.Dense(units=32, activation='relu', input_shape=[1]),   # Primo layer
    keras.layers.Dense(units=16, activation='relu'),                    # Secondo hidden layer
    keras.layers.Dense(units=1)                                         # Output layer
])

In [ ]:
# 3) Compilazione del modello
model.compile(
    optimizer='adam',                                    # Ottimizzatore Adam
    loss='mse',                                          # Funzione di perdita mse = mean_squared_error per regressione
    metrics=['mae']                                      # Funzione per la metrica la mae cioè mean absolute error
)

# Per stampare un riepilogo dell'architettura del modello deselezionare il commento
model.summary()


In [ ]:
# 4) Costruiamo un dataset di dati per la funzione: y=2*X^2+3
# Fissiamo il numero di dati che costituiranno il mio Dataset
Num_Punti = 1000
# Poichè generemo dei numeri casuali, è suggeribile fissare il seme cioè il SEED
# in questo modo ogni volta che lanciamo il Colab si hanno numeri casuali con lo stesso seme
Seed = 1337
# Inizializziamo il generatore di numeri casuali in TensorFlow fissandone il seme
np.random.seed(Seed)
tf.random.set_seed(Seed)
# Facciamo costruire un array di nome xs composto da Num_Punti casuali tra un valore minimo
# di -50 ed un valore massimo di +50
xs = np.random.uniform(low=-50, high=50, size=Num_Punti)                                        # Array degli input
# Facciamo mescolare casualmente con la funzione shuffle i valori di x
np.random.shuffle(xs)
# Calcoliamo il corrispondente valore y partendo dalla funzione nota  y = 2 x^2 + 0 x + 3
ys = 2 * xs * xs  + 3                                                                           # Array degli output
# Facciamo plottare il risultato, ovvero l'array delle x e quello delle y
plt.plot(xs, ys, 'b.')
plt.show()


In [ ]:
# 5) Addestramento del modello
model.fit(
    xs,                                              # Inputs
    ys,                                              # Target outputs
    epochs=1000                                      # Numero di epoche di training
)

# Possiamo far stampare i valori dei parametri determinati dall'apprendimento - deselezionando il commento
# print(model.get_weights())

In [ ]:
# 7) Predizione di esempio con x=10.0
print("Predizione con x=10.0:", model.predict(np.array([10.0])))

In [ ]:
# 8) Esportazione del modello in formato SavedModel
saved_dir = 'saved_model/1'                     # Directory di destinazione
model.export(saved_dir)                         # Esporta il modello in SavedModel compatibile TF
print(f"Modello esportato in formato SavedModel in: {pathlib.Path(saved_dir).resolve()}")

# Calcolo dell’occupazione di memoria del modello salvato
import os                                       #libreria per la gestione dei path di disco
def get_directory_size(saved_dir):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(saved_dir):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
#    return total_size / (1024 * 1024)          # Dimensione in MB
    return total_size                           # Dimensione in Byte

file_size_savedmodel = get_directory_size(saved_dir)
#print(f"Dimensione del modello SavedModel: {file_size_savedmodel:.2f} MB")
print(f"Dimensione del modello SavedModel: {file_size_savedmodel:.2f} Byte")

In [ ]:
# 9) Configurazione del converter TFLite per il SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model(saved_dir)  # Crea il converter
converter.experimental_enable_resource_variables = True          # Fold delle variabili in costanti
tflite_model = converter.convert()                              # Esegue la conversione

In [ ]:
# 10) Salvataggio del flatbuffer TFLite su disco
tflite_path = pathlib.Path('model_from_savedmodel.tflite')      # Percorso output .tflite
tflite_path.write_bytes(tflite_model)                           # Scrive i byte del modello

# Stampa del percorso e della dimensione del file
print(f"TFLite model salvato in: {tflite_path.resolve()}")
print(f"Dimensione del modello TFLite: {tflite_path.stat().st_size} byte")

In [ ]:
# 11) Inizializzazione dell’interprete TFLite
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))  # Carica il modello TFLite
interpreter.allocate_tensors()                                  # Alloca memoria per tensori

In [ ]:
# 12) Recupero dei dettagli di input e output
input_details  = interpreter.get_input_details()  # Metadati sul tensore di input
output_details = interpreter.get_output_details() # Metadati sul tensore di output
print("Input details:", input_details)
print("Output details:", output_details)

In [ ]:
# 13) Esecuzione dell’inferenza TFLite
to_predict = np.array([[10.0]], dtype=np.float32)             # Dati di input formattati
interpreter.set_tensor(input_details[0]['index'], to_predict) # Carica il tensore di input
interpreter.invoke()                                         # Esegue il modello
tflite_results = interpreter.get_tensor(output_details[0]['index'])  # Estrae il risultato
print("Risultato TFLite per x=10.0:", tflite_results)        # Stampa l’output dell'inferenza